# 🎨 ComfyUI Studio - Google Colab Runtime

Notebook này cấu hình môi trường Google Colab thành GPU Runtime hiệu năng cao cho **ComfyUI Studio**:
- 💾 **Lưu trữ bền vững trên Google Drive** (`MyDrive/ComfyStudio/models`): tải một lần, lần sau mở lên dùng ngay, không mất thời gian tải lại.
- ⚡ **Tự động kiểm tra & bỏ qua (Skip)** nếu file model đã tồn tại trên Drive.
- 🖼️ **Tải sẵn bộ Model Qwen-Image-2.1** (INT8 Diffusion, INT8 Text Encoder Qwen3-VL 8B, BF16 VAE) có thanh tiến trình hiển thị % chi tiết.
- 🎛️ **Giao diện UI Dropdown tương tác**: Dễ dàng tải thêm Model, LoRA, VAE, ControlNet... vào đúng thư mục lưu trữ.
- 🌐 **Mở cổng Cloudflare Tunnel** kết nối trực tiếp đến ComfyUI Studio.



In [ ]:
#@title 1. Kết nối Google Drive & Khởi tạo Thư mục Models
import os
import shutil
from google.colab import drive

# 1. Mount Google Drive
print("🚀 Đang kết nối Google Drive...")
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio"
DRIVE_MODELS = os.path.join(DRIVE_BASE, "models")
DRIVE_OUTPUTS = os.path.join(DRIVE_BASE, "outputs")
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

# Các danh mục model chuẩn của ComfyUI
MODEL_CATEGORIES = [
    "checkpoints",
    "diffusion_models",
    "text_encoders",
    "clip",
    "loras",
    "vae",
    "controlnet",
    "upscale_models",
    "embeddings"
]

print(f"📁 Thư mục lưu ảnh tự động trên Google Drive: {DRIVE_OUTPUTS}")
print("📁 Khởi tạo cấu trúc thư mục lưu trữ models trên Google Drive...")
for cat in MODEL_CATEGORIES:
    path = os.path.join(DRIVE_MODELS, cat)
    os.makedirs(path, exist_ok=True)

# Tạo alias tương thích ngược (checkpoint <-> checkpoints, lora <-> loras, upscaler <-> upscale_models)
legacy_aliases = [
    ("checkpoint", "checkpoints"),
    ("lora", "loras"),
    ("upscaler", "upscale_models")
]
for old_name, new_name in legacy_aliases:
    old_p = os.path.join(DRIVE_MODELS, old_name)
    new_p = os.path.join(DRIVE_MODELS, new_name)
    if not os.path.exists(old_p) and os.path.exists(new_p):
        try:
            os.symlink(new_p, old_p)
        except Exception:
            pass

# Kiểm tra dung lượng Google Drive
try:
    total, used, free = shutil.disk_usage(DRIVE_BASE)
    print(f"✅ Google Drive đã kết nối thành công tại: {DRIVE_BASE}")
    print(f"📊 Dung lượng Drive: Đã dùng {used // (2**30)} GB / Tổng {total // (2**30)} GB (Còn trống {free // (2**30)} GB)")
except Exception as e:
    print(f"✅ Google Drive đã kết nối: {DRIVE_BASE}")


In [ ]:
#@title 2. Cài đặt ComfyUI & Cấu hình đường dẫn Models
import os
import subprocess
import time

COMFY_DIR = "/content/ComfyUI"

# 1. Clone ComfyUI với depth=1 để tối ưu tốc độ
if not os.path.exists(COMFY_DIR):
    print("🚀 Đang clone ComfyUI (phiên bản mới nhất)...")
    !git clone --depth=1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
else:
    print("ℹ️ ComfyUI đã có sẵn tại /content/ComfyUI.")

%cd /content/ComfyUI

# 2. Cài đặt PyTorch, requirements và các công cụ download
print("📦 Đang cài đặt PyTorch và thư viện phụ trợ...")
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu124
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q -r requirements.txt
!pip install -q huggingface_hub tqdm requests ipywidgets

# 3. Tạo cấu hình extra_model_paths.yaml để ComfyUI tự nhận diện toàn bộ models trong Google Drive
extra_paths_config = """
colab_drive:
    base_path: /content/drive/MyDrive/ComfyStudio/models
    checkpoints: checkpoints/
    diffusion_models: diffusion_models/
    unet: diffusion_models/
    text_encoders: text_encoders/
    clip: text_encoders/
    loras: loras/
    vae: vae/
    controlnet: controlnet/
    upscale_models: upscale_models/
    embeddings: embeddings/

colab_drive_legacy:
    base_path: /content/drive/MyDrive/ComfyStudio/models
    checkpoints: checkpoint/
    loras: lora/
    upscale_models: upscaler/
"""
with open('/content/ComfyUI/extra_model_paths.yaml', 'w') as f:
    f.write(extra_paths_config.strip())

# 4. Khởi tạo sẵn tất cả thư mục models bên trong ComfyUI
subdirs = [
    "checkpoints", "diffusion_models", "text_encoders", "clip",
    "loras", "vae", "controlnet", "upscale_models", "embeddings"
]
for s in subdirs:
    os.makedirs(f"/content/ComfyUI/models/{s}", exist_ok=True)

# 5. Cài đặt ComfyUI-Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    print("🔌 Đang cài đặt ComfyUI-Manager...")
    !git clone --depth=1 https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
else:
    print("ℹ️ ComfyUI-Manager đã sẵn sàng.")

print("\n✅ Cài đặt ComfyUI & cấu hình liên kết Google Drive thành công!")


In [ ]:
#@title 📦 3. Cài Đặt Danh Sách Model Tuyển Chọn (Checklist & Trọn bộ Full Package)
#@markdown Chọn các model bạn muốn sử dụng cho ComfyUI. Có thể chọn tải **TRỌN BỘ FULLPACK** trong mục Chế độ hoặc tick chọn riêng từng model.
#@markdown - 🔍 **Chế độ kiểm tra**: Quét Drive xem model nào đã có / chưa có, **tuyệt đối không tải lại file đã có**.
#@markdown - ⚡ **Chống tải trùng thông minh**: Nhận diện tên file gốc, tên đổi, alias và kích thước để bỏ qua chính xác 100%.
#@markdown - 🤝 **Dùng chung Drive**: Lưu tại `MyDrive/ComfyStudio/models`, ComfyUI và Automatic1111 dùng chung hoàn toàn.

# --- 🎯 CHẾ ĐỘ THỰC HIỆN ---
Che_Do = "🔍 1. Kiểm tra trạng thái trên Google Drive (Xem cái nào đã có / chưa có)" #@param ["🔍 1. Kiểm tra trạng thái trên Google Drive (Xem cái nào đã có / chưa có)", "⚡ 2. Tải TRỌN BỘ FULLPACK (Tự động bỏ qua các model đã có trên Drive)", "⬇️ 3. Tải các model đã TICK CHỌN (Tự động bỏ qua các model đã có trên Drive)"]

# --- 🌟 PHẦN 1: CHECKPOINT (SDXL & SD1.5) ---
Checkpoint_Krea2_Lustify_Turbo_SDXL = True #@param {type:"boolean"}
Checkpoint_Realistic_Vision_V6_SD15 = True #@param {type:"boolean"}
Checkpoint_WAI_Illustrious_SDXL = False #@param {type:"boolean"}
Checkpoint_majicMIX_Realistic_SD15 = False #@param {type:"boolean"}
Checkpoint_Pony_Diffusion_V6_XL = False #@param {type:"boolean"}

# --- 🎨 PHẦN 2: LORA (PHONG CÁCH & CHI TIẾT) ---
Lora_Krea_2_NSFW_V4 = True #@param {type:"boolean"}
Lora_ExpressiveH_Style = False #@param {type:"boolean"}
Lora_Breasts_Out_Clothing = False #@param {type:"boolean"}

# --- 🛠️ PHẦN 3: BỘ PHỤ TRỢ (VAE & UPSCALER) ---
VAE_SDXL_Fix = True #@param {type:"boolean"}
Upscaler_4x_UltraSharp = True #@param {type:"boolean"}

# --- 🔐 BẢO MẬT & MÃ HÓA TOKEN ---
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}
Civitai_API_Token = "" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

import os
import sys
import subprocess
from huggingface_hub import hf_hub_download

# Cài đặt aria2 nếu chưa có
!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open('/content/.vault_tokens.json', 'w') as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = '', pwd: str = '') -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists('/content/.vault_tokens.json'):
            cached = json.load(open('/content/.vault_tokens.json'))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ''

Civitai_API_Token = get_vault_token('civitai', Civitai_API_Token, Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token('huggingface', HuggingFace_Token, Mat_Khau_Mo_Khoa)
if Civitai_API_Token or HuggingFace_Token:
    print('🔓 [Bảo Mật] Đã mở khóa thành công API Tokens qua Mật khẩu!')

DRIVE_DIR = '/content/drive/MyDrive/ComfyStudio/models'
COMFY_DIR = '/content/ComfyUI/models'
has_drive = os.path.exists('/content/drive/MyDrive')
TARGET_BASE = DRIVE_DIR if has_drive else COMFY_DIR

# DANH MỤC MODEL CHUẨN TUYỂN CHỌN
CURATED_MODELS = [
    # --- CHECKPOINTS ---
    {
        'id': 'krea2_turbo',
        'group': 'CHECKPOINTS',
        'selected': Checkpoint_Krea2_Lustify_Turbo_SDXL,
        'name': 'Krea 2 Turbo / LUSTIFY! NSFW [SDXL 4-Step]',
        'category': 'checkpoints',
        'filename': 'lustify_workflows_krea_2_sdxl.safetensors',
        'aliases': ['lustify', 'krea2', 'krea_2', 'lustifynsfwcheckpoint'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'model_version_id': '3112728',
        'url': 'https://civitai.com/api/download/models/3112728',
        'desc': 'SDXL Turbo cực nét gen 4 bước (~2-4s)'
    },
    {
        'id': 'realistic_vision_v6',
        'group': 'CHECKPOINTS',
        'selected': Checkpoint_Realistic_Vision_V6_SD15,
        'name': 'Realistic Vision V6.0 B1 [SD1.5]',
        'category': 'checkpoints',
        'filename': 'realisticVisionV60B1_v51VAE.safetensors',
        'aliases': ['realisticvision', 'realistic_vision', 'v51hypervae', 'v51vae'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'model_version_id': '130072',
        'url': 'https://civitai.com/api/download/models/130072?type=Model&format=SafeTensor&size=pruned&fp=fp16',
        'alt_hf': {'repo_id': 'SG161222/Realistic_Vision_V5.1_noVAE', 'filename': 'Realistic_Vision_V5.1_fp16-no-ema.safetensors'},
        'desc': 'SD1.5 chụp ảnh người thật photorealistic kinh điển'
    },
    {
        'id': 'wai_illustrious',
        'group': 'CHECKPOINTS',
        'selected': Checkpoint_WAI_Illustrious_SDXL,
        'name': 'WAI-illustrious-SDXL [SDXL Anime/Illustrious]',
        'category': 'checkpoints',
        'filename': 'waiIllustriousSDXL_v170.safetensors',
        'aliases': ['waiillustrious', 'wai-illustrious', 'illustrious_sdxl'],
        'min_size_mb': 2000,
        'source_type': 'civitai',
        'model_version_id': '2883731',
        'url': 'https://civitai.com/api/download/models/2883731',
        'desc': 'SDXL minh họa anime & manga đỉnh cao'
    },
    {
        'id': 'majicmix_realistic',
        'group': 'CHECKPOINTS',
        'selected': Checkpoint_majicMIX_Realistic_SD15,
        'name': 'majicMIX realistic 麦橘写实 [SD1.5]',
        'category': 'checkpoints',
        'filename': 'majicmixRealistic_v7.safetensors',
        'aliases': ['majicmix', 'majicmixrealistic'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'model_version_id': '176425',
        'url': 'https://civitai.com/api/download/models/176425',
        'desc': 'SD1.5 chân dung Châu Á đẹp tự nhiên'
    },
    {
        'id': 'pony_diffusion_v6',
        'group': 'CHECKPOINTS',
        'selected': Checkpoint_Pony_Diffusion_V6_XL,
        'name': 'Pony Diffusion V6 XL [SDXL Pony]',
        'category': 'checkpoints',
        'filename': 'ponyDiffusionV6XL_v6StartWithThisOne.safetensors',
        'aliases': ['ponydiffusion', 'pony_v6', 'ponydiffusionv6xl'],
        'min_size_mb': 2000,
        'source_type': 'civitai',
        'model_version_id': '290640',
        'url': 'https://civitai.com/api/download/models/290640',
        'desc': 'SDXL Pony V6 đa dạng phong cách & art'
    },
    # --- LORAS ---
    {
        'id': 'lora_krea_2_nsfw',
        'group': 'LORAS',
        'selected': Lora_Krea_2_NSFW_V4,
        'name': 'LoRA Krea 2 NSFW V4',
        'category': 'loras',
        'filename': 'krea_2_nsfw_v4.safetensors',
        'aliases': ['knp_000003000', 'krea_2_nsfw', 'knpv4'],
        'min_size_mb': 10,
        'source_type': 'civitai',
        'model_version_id': '3147117',
        'url': 'https://civitai.com/api/download/models/3147117',
        'desc': 'LoRA tăng chi tiết cho Krea 2 Turbo'
    },
    {
        'id': 'lora_expressive_h',
        'group': 'LORAS',
        'selected': Lora_ExpressiveH_Style,
        'name': 'LoRA ExpressiveH (Hentai LoRa Style)',
        'category': 'loras',
        'filename': 'Expressive_H-000001.safetensors',
        'aliases': ['expressive_h', 'expressiveh'],
        'min_size_mb': 10,
        'source_type': 'civitai',
        'model_version_id': '382152',
        'url': 'https://civitai.com/api/download/models/382152',
        'desc': 'LoRA phong cách anime cảm xúc cao'
    },
    {
        'id': 'lora_breasts_out',
        'group': 'LORAS',
        'selected': Lora_Breasts_Out_Clothing,
        'name': 'LoRA Breasts Out Fashion Clothing',
        'category': 'loras',
        'filename': '2BoutOval_K2v3_400.safetensors',
        'aliases': ['2boutoval', 'breasts_out', 'breastsout'],
        'min_size_mb': 5,
        'source_type': 'civitai',
        'model_version_id': '3134691',
        'url': 'https://civitai.com/api/download/models/3134691',
        'desc': 'LoRA phong cách thời trang gợi cảm'
    },
    # --- UTILITIES ---
    {
        'id': 'vae_sdxl',
        'group': 'UTILITIES',
        'selected': VAE_SDXL_Fix,
        'name': 'SDXL VAE (Fix Màu & Khử Mờ)',
        'category': 'vae',
        'filename': 'sdxl_vae.safetensors',
        'aliases': ['sdxl_vae', 'sdxlvae'],
        'min_size_mb': 50,
        'source_type': 'hf',
        'repo_id': 'stabilityai/sdxl-vae',
        'hf_filename': 'sdxl_vae.safetensors',
        'desc': 'VAE chuẩn SDXL cân bằng màu'
    },
    {
        'id': 'upscaler_4x_ultrasharp',
        'group': 'UTILITIES',
        'selected': Upscaler_4x_UltraSharp,
        'name': 'Upscaler 4x-UltraSharp',
        'category': 'upscale_models',
        'filename': '4x-UltraSharp.pth',
        'aliases': ['4x-ultrasharp', '4x_ultrasharp', 'ultrasharp'],
        'min_size_mb': 10,
        'source_type': 'hf',
        'repo_id': 'lokcx/4x-Ultrasharp',
        'hf_filename': '4x-UltraSharp.pth',
        'desc': 'Nâng nét ảnh lên độ phân giải 4K'
    }
]

# HÀM QUÉT MODEL THÔNG MINH - CHỐNG TẢI TRÙNG 100%
def find_existing_model(target_base, category, primary_name, aliases=None, min_mb=10):
    folder = os.path.join(target_base, category)
    if not os.path.exists(folder):
        return None, 0
    
    candidates = [primary_name.lower()]
    if aliases:
        for a in aliases:
            candidates.append(a.lower())
            
    min_bytes = min_mb * 1024 * 1024
    
    try:
        for fname in os.listdir(folder):
            full_path = os.path.join(folder, fname)
            if not os.path.isfile(full_path):
                continue
            if fname.endswith('.aria2'):
                continue
                
            sz = os.path.getsize(full_path)
            if sz < min_bytes:
                continue
                
            lower_fname = fname.lower()
            # 1. Khớp chính xác tên candidate
            for cand in candidates:
                if cand == lower_fname or cand in lower_fname:
                    return full_path, sz
    except Exception:
        pass
        
    return None, 0

def link_model_file(src_file, dst_file):
    if not os.path.exists(src_file):
        return False
    try:
        os.makedirs(os.path.dirname(dst_file), exist_ok=True)
        if os.path.exists(dst_file) and not os.path.islink(dst_file) and os.path.samefile(src_file, dst_file):
            return True
        if not os.path.exists(dst_file):
            os.symlink(src_file, dst_file)
            return True
    except Exception:
        pass
    return False

# ==========================================================================
# CHẾ ĐỘ 1: KIỂM TRA TRẠNG THÁI TRÊN GOOGLE DRIVE (KHÔNG TẢI GÌ)
# ==========================================================================
if Che_Do.startswith('🔍'):
    print('=' * 88)
    print('📊 BẢNG KIỂM TRA MODEL TRÊN GOOGLE DRIVE CHO ComfyUI')
    print(f'📂 Thư mục: {TARGET_BASE}')
    print('=' * 88)
    print(f"{'STT':<4} | {'TRẠNG THÁI':<12} | {'DANH MỤC':<15} | {'DUNG LƯỢNG':<10} | {'TÊN MODEL'}")
    print('-' * 88)
    
    count_exist = 0
    total_exist_bytes = 0
    current_grp = ''
    
    for idx, item in enumerate(CURATED_MODELS, 1):
        grp = item['group']
        if grp != current_grp:
            print(f'\n--- 🔹 NHÓM: {grp} ---')
            current_grp = grp
            
        existing_file, fsize = find_existing_model(TARGET_BASE, item['category'], item['filename'], item.get('aliases'), item['min_size_mb'])
        target_dest_file = os.path.join(COMFY_DIR, item['category'], item['filename'])
        
        if existing_file:
            count_exist += 1
            total_exist_bytes += fsize
            sz_str = f"{fsize / (1024**3):.2f} GB" if fsize >= 1024**3 else f"{fsize / (1024**2):.1f} MB"
            status_str = '✅ ĐÃ CÓ'
            fname_display = os.path.basename(existing_file)
            link_model_file(existing_file, target_dest_file)
        else:
            sz_str = f"~{item['min_size_mb']} MB"
            status_str = '❌ CHƯA CÓ'
            fname_display = item['filename']
            
        print(f"{idx:<4} | {status_str:<12} | {item['category']:<15} | {sz_str:<10} | {item['name']}")
        print(f"     ↳ File trên đĩa: {fname_display} ({item['desc']})")
    
    total_gb = total_exist_bytes / (1024**3)
    print('=' * 88)
    print(f'📈 TỔNG KẾT: Đã có {count_exist}/{len(CURATED_MODELS)} model trên Drive (Dung lượng: {total_gb:.2f} GB). Còn thiếu: {len(CURATED_MODELS) - count_exist} model.')
    print('💡 HƯỚNG DẪN:')
    print('   - Tải tất cả model còn thiếu: Đổi [Che_Do] sang "⚡ 2. Tải TRỌN BỘ FULLPACK" rồi bấm Play!')
    print('   - Tải model tick chọn: Đổi [Che_Do] sang "⬇️ 3. Tải các model đã TICK CHỌN" rồi bấm Play!')
    print('=' * 88)

# ==========================================================================
# CHẾ ĐỘ 2 & 3: TIẾN TRÌNH TẢI MODEL (TỰ ĐỘNG BỎ QUA FILE ĐÃ CÓ)
# ==========================================================================
else:
    is_fullpack = 'FULLPACK' in Che_Do
    targets = [m for m in CURATED_MODELS if is_fullpack or m['selected']]
    
    print('=' * 88)
    mode_name = 'TRỌN BỘ FULLPACK (Tất cả model)' if is_fullpack else 'CÁC MODEL ĐÃ TICK CHỌN'
    print(f'🚀 BẮT ĐẦU TIẾN TRÌNH TẢI: {mode_name}')
    print(f'🎯 Thư mục lưu trữ: {TARGET_BASE}')
    print('=' * 88)
    
    if not targets:
        print('⚠️ Bạn chưa tick chọn model nào! Hãy tick chọn model hoặc chọn chế độ FULLPACK.')
    
    for item in targets:
        category = item['category']
        filename = item['filename']
        min_bytes = item['min_size_mb'] * 1024 * 1024
        dest_dir = os.path.join(TARGET_BASE, category)
        os.makedirs(dest_dir, exist_ok=True)
        dest_file = os.path.join(dest_dir, filename)
        target_dest_file = os.path.join(COMFY_DIR, category, filename)
        
        # 1. KIỂM TRA THÔNG MINH XEM ĐÃ CÓ FILE TRÊN DRIVE CHƯA
        existing_file, fsize = find_existing_model(TARGET_BASE, category, filename, item.get('aliases'), item['min_size_mb'])
        if existing_file:
            sz = fsize / (1024**3)
            print(f"⏩ [BỎ QUA - ĐÃ CÓ SẴN] {item['name']}")
            print(f"   Đã tìm thấy file: {os.path.basename(existing_file)} ({sz:.2f} GB) trên Drive -> BỎ QUA KHÔNG TẢI LẠI!")
            link_model_file(existing_file, target_dest_file)
            continue
            
        print(f"\n⬇️ [ĐANG TẢI MỚI] {item['name']}...")
        print(f"   Lưu vào: {dest_file}")
        
        # 2. Tải tùy theo nguồn (Civitai hoặc HuggingFace)
        if item['source_type'] == 'civitai':
            url = item['url']
            if Civitai_API_Token and 'token=' not in url:
                url += f'?token={Civitai_API_Token}' if '?' not in url else f'&token={Civitai_API_Token}'
                
            cmd = [
                'aria2c', '-c', '-x', '16', '-s', '16', '-k', '1M',
                '--file-allocation=none',
                '--header=User-Agent: Mozilla/5.0',
                '-d', dest_dir,
                '-o', filename,
                url
            ]
            ret = subprocess.run(cmd)
            if ret.returncode != 0 and 'alt_hf' in item:
                print('⚠️ Nguồn Civitai gặp sự cố, tự động chuyển sang HuggingFace dự phòng...')
                hf = item['alt_hf']
                try:
                    hf_hub_download(repo_id=hf['repo_id'], filename=hf['filename'], local_dir=dest_dir, local_dir_use_symlinks=False)
                    tmp = os.path.join(dest_dir, hf['filename'])
                    if tmp != dest_file and os.path.exists(tmp):
                        os.rename(tmp, dest_file)
                except Exception as err:
                    print(f'❌ HF dự phòng lỗi: {err}')
                    
        elif item['source_type'] == 'hf':
            try:
                dl_kwargs = {
                    'repo_id': item['repo_id'],
                    'filename': item['hf_filename'],
                    'local_dir': dest_dir,
                    'local_dir_use_symlinks': False
                }
                if HuggingFace_Token:
                    dl_kwargs['token'] = HuggingFace_Token
                dl_path = hf_hub_download(**dl_kwargs)
                if dl_path != dest_file and os.path.exists(dl_path):
                    if os.path.exists(dest_file):
                        os.remove(dest_file)
                    os.rename(dl_path, dest_file)
            except Exception as e:
                print(f"❌ Lỗi tải HuggingFace {filename}: {e}")
                
        # 3. Xác thực sau tải
        existing_file, fsize = find_existing_model(TARGET_BASE, category, filename, item.get('aliases'), item['min_size_mb'])
        if existing_file:
            sz = fsize / (1024**3)
            print(f"✅ Tải thành công: {filename} ({sz:.2f} GB)")
            link_model_file(existing_file, target_dest_file)
        else:
            print(f'⚠️ Cảnh báo: Tải {filename} chưa hoàn tất hoặc file bị lỗi.')
            
    print('\n🎉 Hoàn thành kiểm tra & cập nhật model cho ComfyUI!')


In [ ]:
#@title 📥 4. Tải Thêm Model Tùy Chọn (Check Trùng Trước Khi Tải)
#@markdown Tải thêm bất kỳ Model nào từ **Civitai** (civitai.com, civitai.red), **HuggingFace**, hoặc Direct Link:
#@markdown - 🔍 **Kiểm tra trước khi tải**: Biết ngay model đã có trong Google Drive chưa, tránh tải trùng.
#@markdown - 🛑 **Tự động Chặn Tải Trùng**: Nếu file đã có sẵn trên Drive, hệ thống sẽ từ chối tải lại để bảo vệ dung lượng.
#@markdown - ⚡ **Bypass 403 Forbidden Civitai**: Tự động nhận diện `modelVersionId` và tải qua API token.
#@markdown - 🔗 **Tự động liên kết**: Tự động tạo symlink vào đúng thư mục của ComfyUI.

# --- CẤU HÌNH TẢI MODEL ---
Thu_Muc_Luu = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "diffusion_models", "text_encoders", "embeddings"]
Link_Model = "" #@param {type:"string"}
Doi_Ten_File_Tuy_Chon = "" #@param {type:"string"}

# --- THAO TÁC THỰC HIỆN ---
Thao_Tac = "🔍 1. Kiểm tra xem model này đã có trên Google Drive chưa" #@param ["🔍 1. Kiểm tra xem model này đã có trên Google Drive chưa", "⬇️ 2. Bắt đầu tải (Tự động CHẶN nếu đã có sẵn)"]

# --- MẬT KHẨU MÃ HÓA & API TOKENS ---
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}
Civitai_API_Token = "" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

Civitai_API_Token = get_vault_token("civitai", Civitai_API_Token, Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token("huggingface", HuggingFace_Token, Mat_Khau_Mo_Khoa)


import os
import re
import sys
import json
import urllib.request
import subprocess

# Cài đặt aria2 nếu chưa có
!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
COMFY_DIR = "/content/ComfyUI/models"
has_drive = os.path.exists("/content/drive/MyDrive")
TARGET_BASE = DRIVE_DIR if has_drive else COMFY_DIR

def resolve_model_info(url, token=""):
    headers = {"User-Agent": "Mozilla/5.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    
    # 1. Kiểm tra Civitai URL
    if "civitai.com" in url or "civitai.red" in url:
        ver_match = re.search(r'modelVersionId=(\d+)', url)
        if ver_match:
            ver_id = ver_match.group(1)
            api_url = f"https://civitai.com/api/v1/model-versions/{ver_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    files = data.get("files", [])
                    fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                    size_kb = files[0].get("sizeKB", 0) if files else 0
                    dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                    if token:
                        dl_url += f"?token={token}"
                    return fname, dl_url, size_kb / 1024
            except Exception as e:
                print(f"⚠️ Không đọc được API Civitai: {e}")
                return f"{ver_id}.safetensors", f"https://civitai.com/api/download/models/{ver_id}?token={token}", 0
        
        model_match = re.search(r'/models/(\d+)', url)
        if model_match:
            model_id = model_match.group(1)
            api_url = f"https://civitai.com/api/v1/models/{model_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    versions = data.get("modelVersions", [])
                    if versions:
                        ver = versions[0]
                        ver_id = ver.get("id")
                        files = ver.get("files", [])
                        fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                        size_kb = files[0].get("sizeKB", 0) if files else 0
                        dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                        if token:
                            dl_url += f"?token={token}"
                        return fname, dl_url, size_kb / 1024
            except Exception as e:
                print(f"⚠️ Không đọc được API Civitai: {e}")
                return f"{model_id}.safetensors", url, 0
                
    # 2. HuggingFace hoặc Direct link
    clean_url = url.split("?")[0]
    fname = os.path.basename(clean_url)
    return fname, url, 0

clean_link = Link_Model.strip()
if not clean_link:
    print("⚠️ VUI LÒNG DÁN LINK MODEL VÀO Ô [Link_Model] Ở TRÊN ĐỂ TIẾP TỤC.")
else:
    detected_name, download_url, estimated_mb = resolve_model_info(clean_link, Civitai_API_Token)
    final_filename = Doi_Ten_File_Tuy_Chon.strip() if Doi_Ten_File_Tuy_Chon.strip() else detected_name
    if not final_filename:
        final_filename = "downloaded_model.safetensors"
        
    target_dir = os.path.join(TARGET_BASE, Thu_Muc_Luu)
    os.makedirs(target_dir, exist_ok=True)
    dest_path = os.path.join(target_dir, final_filename)
    comfy_path = os.path.join(COMFY_DIR, Thu_Muc_Luu, final_filename)
    
    is_already_exist = os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024
    
    # ----------------------------------------------------------------------
    # THAO TÁC 1: KIỂM TRA TRƯỚC KHI TẢI
    # ----------------------------------------------------------------------
    if Thao_Tac.startswith("🔍"):
        print("=" * 75)
        print("🔍 KẾT QUẢ KIỂM TRA MODEL TRÊN GOOGLE DRIVE")
        print("=" * 75)
        print(f"📂 Danh mục lưu:   {Thu_Muc_Luu}/")
        print(f"📄 Tên file model: {final_filename}")
        print(f"📍 Đường dẫn đích: {dest_path}")
        print("-" * 75)
        
        if is_already_exist:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print(f"📊 Trạng thái:     ✅ ĐÃ CÓ TRÊN GOOGLE DRIVE! ({sz_str})")
            print("=" * 75)
            print("🛑 KẾT LUẬN: BẠN KHÔNG CẦN TẢI LẠI! Model này đã có sẵn đầy đủ.")
            print("   Hệ thống sẽ bảo vệ Google Drive của bạn khỏi bị tải trùng lãng phí.")
            if not os.path.exists(comfy_path):
                try:
                    os.makedirs(os.path.dirname(comfy_path), exist_ok=True)
                    os.symlink(dest_path, comfy_path)
                    print(f"   🔗 Đã tự động tạo Symlink vào ComfyUI: {comfy_path}")
                except Exception:
                    pass
            else:
                print(f"   🔗 ComfyUI đã sẵn sàng nhận diện model này.")
        else:
            est_str = f"~{estimated_mb:.1f} MB" if estimated_mb > 0 else "Chưa rõ"
            print(f"📊 Trạng thái:     ❌ CHƯA CÓ TRÊN GOOGLE DRIVE (Dung lượng dự kiến: {est_str})")
            print("=" * 75)
            print("💡 BẠN CÓ THỂ TẢI MODEL NÀY:")
            print("   1. Đổi mục [Thao_Tac] ở trên sang: '⬇️ 2. Bắt đầu tải (Tự động CHẶN nếu đã có sẵn)'")
            print("   2. Bấm nút Play (▶️) để bắt đầu tải đa luồng tốc độ cao bằng aria2c!")
            print("=" * 75)
            
    # ----------------------------------------------------------------------
    # THAO TÁC 2: BẮT ĐẦU TẢI
    # ----------------------------------------------------------------------
    else:
        if is_already_exist:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print("=" * 75)
            print(f"🛑 CHẶN TẢI TRÙNG LẶP: Model '{final_filename}' ({sz_str}) ĐÃ CÓ TRÊN GOOGLE DRIVE!")
            print("   Hủy lệnh tải để không tốn thời gian và dung lượng lưu trữ.")
            if not os.path.exists(comfy_path):
                try:
                    os.makedirs(os.path.dirname(comfy_path), exist_ok=True)
                    os.symlink(dest_path, comfy_path)
                    print(f"   🔗 Đã liên kết symlink sang ComfyUI: {comfy_path}")
                except Exception:
                    pass
            print("=" * 75)
        else:
            print("=" * 75)
            print(f"⬇️ BẮT ĐẦU TẢI: {final_filename}")
            print(f"   URL: {clean_link}")
            print(f"   Lưu vào: {dest_path}")
            print("=" * 75)
            
            cmd = [
                "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
                "--file-allocation=none",
                "--header=User-Agent: Mozilla/5.0",
                "-d", target_dir,
                "-o", final_filename,
                download_url
            ]
            subprocess.run(cmd)
            
            if os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024:
                sz_bytes = os.path.getsize(dest_path)
                sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
                print("=" * 75)
                print(f"🎉 TẢI THÀNH CÔNG: {final_filename} ({sz_str})")
                if not os.path.exists(comfy_path):
                    try:
                        os.makedirs(os.path.dirname(comfy_path), exist_ok=True)
                        os.symlink(dest_path, comfy_path)
                        print(f"🔗 Đã liên kết symlink sang ComfyUI: {comfy_path}")
                    except Exception:
                        pass
                print("=" * 75)
            else:
                print(f"❌ Tải thất bại hoặc file không hợp lệ: {dest_path}")


In [ ]:
#@title 3B. Tải bộ Model Qwen-Image-2.1 (Tùy chọn - Dành cho ai muốn dùng Qwen)
#@markdown Tải bộ 3 model Qwen-Image-2.1 phục vụ workflow Text-to-Image (Lưu ý: Model nặng 30GB, gen ~20p):
#@markdown - 1. **Diffusion Model (INT8)**: `diffusion_models/qwen_image_2.1_int8_convrot.safetensors` (~12.3 GB)
#@markdown - 2. **Text Encoder Qwen3-VL 8B (INT8)**: `text_encoders/qwen3vl_8b_int8_convrot.safetensors` (~8.8 GB)
#@markdown - 3. **VAE (BF16)**: `vae/qwen_image_2.1_vae_bf16.safetensors` (~254 MB)
#@markdown
#@markdown *Tiến trình hiển thị % dung lượng tải thực tế. Nếu file đã có trên Drive sẽ tự động bỏ qua.*

import os
import sys
import inspect
from huggingface_hub import hf_hub_download

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
COMFY_DIR = "/content/ComfyUI/models"
has_drive = os.path.exists(DRIVE_DIR)

# Danh sách 3 model cần tải theo cấu trúc chuẩn ComfyUI
QWEN_MODELS = [
    {
        "desc": "1. Diffusion Model (INT8)",
        "repo_id": "Comfy-Org/Qwen-Image-2.1",
        "category": "diffusion_models",
        "rel_path": "diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        "basename": "qwen_image_2.1_int8_convrot.safetensors",
        "min_size_mb": 5000 # tối thiểu ~5GB
    },
    {
        "desc": "2. Text Encoder Qwen3-VL 8B (INT8)",
        "repo_id": "Comfy-Org/Qwen-Image-2.1",
        "category": "text_encoders",
        "rel_path": "text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        "basename": "qwen3vl_8b_int8_convrot.safetensors",
        "min_size_mb": 4000 # tối thiểu ~4GB
    },
    {
        "desc": "3. VAE (BF16)",
        "repo_id": "Comfy-Org/Qwen-Image-2.1",
        "category": "vae",
        "rel_path": "vae/qwen_image_2.1_vae_bf16.safetensors",
        "basename": "qwen_image_2.1_vae_bf16.safetensors",
        "min_size_mb": 100 # tối thiểu ~100MB
    }
]

# Thư mục gốc lưu trữ: ưu tiên Google Drive để lần sau không phải tải lại
TARGET_BASE = DRIVE_DIR if has_drive else COMFY_DIR
print(f"🎯 Vị trí lưu trữ mục tiêu: {TARGET_BASE}")
if has_drive:
    print("💡 Đã phát hiện Google Drive. Model sẽ được lưu vào Drive để dùng vĩnh viễn!")
else:
    print("⚠️ Chưa mount Google Drive! Model sẽ được lưu tạm thời trên Colab.")

for item in QWEN_MODELS:
    desc = item["desc"]
    repo_id = item["repo_id"]
    category = item["category"]
    rel_path = item["rel_path"]
    basename = item["basename"]
    min_bytes = item["min_size_mb"] * 1024 * 1024

    drive_file = os.path.join(DRIVE_DIR, category, basename) if has_drive else None
    comfy_file = os.path.join(COMFY_DIR, category, basename)

    # 1. Kiểm tra nếu đã có trên Google Drive
    if drive_file and os.path.exists(drive_file) and os.path.getsize(drive_file) >= min_bytes:
        size_gb = os.path.getsize(drive_file) / (1024 ** 3)
        print(f"⏭️  [ĐÃ CÓ TRÊN DRIVE] {desc} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
        # Đảm bảo ComfyUI có symlink đến file trong Drive
        if not os.path.exists(comfy_file):
            try:
                os.makedirs(os.path.dirname(comfy_file), exist_ok=True)
                os.symlink(drive_file, comfy_file)
                print(f"    🔗 Đã liên kết symlink sang ComfyUI: {comfy_file}")
            except Exception:
                pass
        continue

    # 2. Kiểm tra nếu đã có trên local ComfyUI
    if os.path.exists(comfy_file) and os.path.getsize(comfy_file) >= min_bytes:
        size_gb = os.path.getsize(comfy_file) / (1024 ** 3)
        print(f"⏭️  [ĐÃ CÓ TRÊN COMFYUI] {desc} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
        continue

    # 3. Tải model với tiến trình % (tqdm/huggingface_hub)
    print(f"\n⬇️  Đang tải {desc}: {basename}...")
    print(f"   Repo: {repo_id} | Thư mục đích: {category}/")

    # Xác định tham số tương thích cho hf_hub_download
    download_kwargs = {
        "repo_id": repo_id,
        "filename": rel_path,
        "local_dir": TARGET_BASE
    }
    sig = inspect.signature(hf_hub_download).parameters
    if "local_dir_use_symlinks" in sig:
        download_kwargs["local_dir_use_symlinks"] = False

    # Thực hiện tải (có hiển thị % tiến trình, dung lượng và tốc độ MB/s qua tqdm)
    downloaded_path = hf_hub_download(**download_kwargs)
    print(f"✅ Đã tải xong: {desc}")

    # Nếu tải vào Drive, tạo symlink sang ComfyUI để ComfyUI thấy ngay lập tức
    if has_drive and os.path.exists(drive_file) and not os.path.exists(comfy_file):
        try:
            os.makedirs(os.path.dirname(comfy_file), exist_ok=True)
            os.symlink(drive_file, comfy_file)
            print(f"   🔗 Đã liên kết symlink sang ComfyUI: {comfy_file}")
        except Exception:
            pass

print("\n🎉 Hoàn thành kiểm tra & tải trọn bộ model Qwen-Image-2.1!")


In [ ]:
#@title 5. Khởi chạy ComfyUI & Mở Tunnel (Ngrok / LocalTunnel)
#@markdown Cloudflare thường xuyên chặn ComfyUI (Lỗi 403 Forbidden). Hãy dùng **LocalTunnel** (miễn phí, không cần tài khoản) hoặc **Ngrok** (nhanh, ổn định, cần authtoken).

tunnel_type = "localtunnel (Miễn phí, không cần Token)" #@param ["localtunnel (Miễn phí, không cần Token)", "ngrok (Cần Auth Token)"]
ngrok_auth_token = "" #@param {type:"string"}
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

ngrok_auth_token = get_vault_token("ngrok", ngrok_auth_token, Mat_Khau_Mo_Khoa)
if ngrok_auth_token:
    print("🔓 [Bảo Mật] Đã tự động điền Ngrok Auth Token từ Mật khẩu giải mã!")


import subprocess
import time
import re
import os

# 1. Khởi chạy ComfyUI dưới nền
print("🚀 Đang khởi chạy ComfyUI backend (Output tự động lưu vào Google Drive)...")
log_file = open("/content/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    [
        "python", "/content/ComfyUI/main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--preview-method", "auto",
        "--output-directory", "/content/drive/MyDrive/ComfyStudio/outputs"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# 2. Mở Tunnel
if "ngrok" in tunnel_type.lower():
    if not ngrok_auth_token:
        print("❌ BẠN CHƯA NHẬP NGROK AUTH TOKEN!")
        print("Hãy lấy token tại https://dashboard.ngrok.com/get-started/your-authtoken")
    else:
        print("🔗 Đang thiết lập Ngrok Tunnel...")
        !pip install pyngrok -q
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = ngrok_auth_token
        public_url = ngrok.connect(8188).public_url
        print("\n=======================================================")
        print(f"🎉 COMFYUI STUDIO ENDPOINT: {public_url}")
        print("👉 Sao chép URL trên và dán vào ComfyUI Studio -> Settings / Runtime")
        print("=======================================================\n")
else:
    print("🔗 Đang thiết lập LocalTunnel...")
    !npm install -g localtunnel > /dev/null 2>&1
    tunnel_proc = subprocess.Popen(
        ['lt', '--port', '8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in tunnel_proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.loca\.lt', line)
        if match:
            url = match.group(0)
            print("\n=======================================================")
            print(f"🎉 COMFYUI STUDIO ENDPOINT: {url}")
            print("👉 Sao chép URL trên và dán vào ComfyUI Studio -> Settings / Runtime")
            print("=======================================================\n")
            break
